# Notebook 09: Model Export & Deployment Pipeline

**Phase:** Deployment (Thesis Plan Deliverable 4)

**Purpose:** Convert trained PyTorch models through the deployment pipeline:
PyTorch → ONNX → ONNX Runtime Mobile (Android deployment target).

**Note:** TFLite conversion (via onnx2tf) is NOT used. The Android target uses
ONNX Runtime Mobile, which runs ONNX models directly — no TF/TFLite bridge needed.

## Overview

The deployment sequence follows the thesis plan but uses ONNX Runtime Mobile
for Android deployment instead of TFLite (no TF/TFLite bridge needed):

```
PyTorch
   ↓
ONNX Export
   ↓
ONNX Validation (output vs PyTorch baseline)
   ↓
ONNX Latency Benchmarking
   ↓
ONNX Runtime Mobile → Android Integration
```

## Workflow

1. Load trained PyTorch models (allergen + semantic)
2. Export to ONNX format (dynamic batch)
3. **Validate ONNX output against PyTorch baseline** (new)
4. Benchmark PyTorch and ONNX inference latency
5. Generate deployment report with ONNX validation results

**Rationale:** onnx2tf removed — the full ONNX→TF→TFLite conversion chain
was scaffold-only and never implemented. ONNX Runtime Mobile is a simpler,
production-ready path for Android deployment.

In [1]:
import os
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.model_utils import load_model_and_tokenizer
from utils.deployment_utils import (
    pytorch_to_onnx,
    validate_onnx_output,
    measure_model_size,
    get_model_size_report,
    benchmark_latency,
    benchmark_onnx_latency,
    print_deployment_status,
    check_deployment_requirements,
)
from utils.data_utils import get_data_directories

print("Imports successful.")

Imports successful.


## 1. Check Deployment Environment

In [2]:
print_deployment_status()

dirs = get_data_directories()
export_dir = os.path.join(dirs['base'], 'models', 'exported')
os.makedirs(export_dir, exist_ok=True)
print(f"\nExport target directory: {export_dir}")

I0000 00:00:1782422047.476076  115570 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782422048.708073  115570 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


📦 Deployment Dependencies Status
  ✅ torch
  ✅ transformers
  ✅ onnx
  ✅ onnxruntime
  ✅ tensorflow
  ✅ tflite

Deployment target: ONNX Runtime Mobile (Android)
  No TFLite conversion required — onnx2tf removed.

Export target directory: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/exported


## 2. Load Trained Models

Load both the allergen detection model and (if available) the semantic classification model.

In [3]:
# Load allergen detection model
allergen_model_path = "../models/mobilebert_allergen_final/"
if os.path.exists(allergen_model_path):
    allergen_model, allergen_tokenizer, device = load_model_and_tokenizer(allergen_model_path)
    print(f"✅ Allergen model loaded from {allergen_model_path}")
else:
    print(f"❌ Allergen model not found at {allergen_model_path}")
    allergen_model = None

# Load semantic classification model (if exists)
semantic_model_path = "../models/mobilebert_semantic_final/"
if os.path.exists(semantic_model_path):
    semantic_model, semantic_tokenizer, _ = load_model_and_tokenizer(semantic_model_path)
    print(f"✅ Semantic model loaded from {semantic_model_path}")
else:
    print(f"⚠️  Semantic model not yet trained — skip or train with Notebook 05 first.")
    semantic_model = None

✅ Allergen model loaded from ../models/mobilebert_allergen_final/
✅ Semantic model loaded from ../models/mobilebert_semantic_final/


## 3. Measure Baseline Model Sizes

In [4]:
print("Baseline Model Sizes (PyTorch):")
print("=" * 40)

# Try multiple weight file formats (safetensors is the modern default)
for label, model_path in [("Allergen", allergen_model_path), ("Semantic", semantic_model_path)]:
    if not os.path.exists(model_path):
        print(f"  ⚠️  {label}: directory not found at {model_path}")
        continue

    # Check for weight files in order of preference
    weight_files = [
        ("model.safetensors", os.path.join(model_path, "model.safetensors")),
        ("pytorch_model.bin", os.path.join(model_path, "pytorch_model.bin")),
    ]
    found = False
    for wt_name, wt_path in weight_files:
        if os.path.exists(wt_path):
            print(f"\n{label} model ({wt_name}):")
            print(get_model_size_report(wt_path))
            found = True
            break

    if not found:
        print(f"\n  ⚠️  {label}: No model weights found (checked: {[k for k, v in weight_files]})")
        print(f"      Files in directory: {os.listdir(model_path)}")

Baseline Model Sizes (PyTorch):

Allergen model (model.safetensors):
📦 Model: model.safetensors
   Size: 93.92 MB (96174.2 KB, 98,482,432 bytes)

Semantic model (model.safetensors):
📦 Model: model.safetensors
   Size: 93.99 MB (96244.4 KB, 98,554,260 bytes)


## 4. ONNX Export

Export the PyTorch model(s) to ONNX format with dynamic batch support.

In [5]:
if allergen_model is not None:
    # Prepare dummy input for tracing
    dummy_text = ["milk, sugar, wheat flour, salt"]
    encoded = allergen_tokenizer(dummy_text, padding=True, truncation=True, max_length=221, return_tensors="pt")
    
    onnx_dir = os.path.join(export_dir, "allergen_model")
    try:
        onnx_path = pytorch_to_onnx(
            model=allergen_model,
            tokenizer=allergen_tokenizer,
            dummy_input_ids=encoded['input_ids'].numpy(),
            dummy_attention_mask=encoded['attention_mask'].numpy(),
            output_path=onnx_dir,
            model_name="allergen_model",
            opset_version=18,
        )
        print(f"ONNX export path: {onnx_path}")
        
        # Confirm file was actually created
        if os.path.exists(onnx_path):
            size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
            print(f"✅ ONNX file size: {size_mb:.2f} MB")
        else:
            print("❌ ONNX file was not created despite no error.")
    except Exception as e:
        print(f"❌ ONNX export failed: {e}")
        onnx_path = None
else:
    print("Skipping ONNX export — no model loaded.")

[deployment_utils] ONNX export config saved to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/exported/allergen_model
[deployment_utils] Exporting model to ONNX...
[deployment_utils] Target: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/exported/allergen_model/allergen_model.onnx
[deployment_utils] ONNX model exported to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/exported/allergen_model/allergen_model.onnx
ONNX export path: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/exported/allergen_model/allergen_model.onnx
✅ ONNX file size: 94.47 MB


## 5. ONNX Output Validation

Validate that the exported ONNX model produces outputs consistent with the PyTorch
baseline. ONNX Runtime runs the .onnx file on CPU (no GPU), so tiny numeric
differences from floating-point op reordering are expected.

In [6]:
if allergen_model is not None and onnx_path is not None and os.path.exists(onnx_path):
    print("=" * 50)
    print("ONNX Output Validation (vs PyTorch baseline)")
    print("=" * 50)
    
    val = validate_onnx_output(
        onnx_path=onnx_path,
        pytorch_model=allergen_model,
        tokenizer=allergen_tokenizer,
        test_text="milk powder, sugar, cocoa butter, soy lecithin, vanilla extract",
        tolerance=5e-3,
    )
    
    if val["within_tolerance"]:
        print(f"\n✅ ONNX model validated — max diff {val['max_diff']:.6e} is within tolerance.")
    else:
        print(f"\n⚠️  ONNX output exceeds tolerance ({val['max_diff']:.6e} > {val['tolerance']})")
else:
    print("Skipping validation — no ONNX model available.")

ONNX Output Validation (vs PyTorch baseline)
[deployment_utils] ONNX output validation
[deployment_utils]   Max diff vs PyTorch:  1.144409e-05
[deployment_utils]   Mean diff vs PyTorch: 4.887581e-06
[deployment_utils]   Within tolerance (0.005): ✅ YES

✅ ONNX model validated — max diff 1.144409e-05 is within tolerance.


## 6. Latency Benchmarking

Benchmark inference latency for both PyTorch (baseline) and ONNX Runtime (deployment target).

In [7]:
if allergen_model is not None:
    # Benchmark text for inference
    benchmark_texts = [
        "milk powder, sugar, cocoa butter, soy lecithin, salt",
        "wheat flour, water, sugar, yeast, salt",
        "enriched flour, sugar, vegetable oil, corn syrup, leavening",
    ]
    
    # ── PyTorch Latency ──
    print("Inference Latency (PyTorch, CPU):")
    print("-" * 40)
    pt_results = benchmark_latency(
        model=allergen_model,
        tokenizer=allergen_tokenizer,
        test_texts=benchmark_texts,
        max_length=221,
        n_runs=100,
        device="cpu",
    )
    print(f"  Mean:   {pt_results['mean_ms']:.1f} ms")
    print(f"  Median: {pt_results['median_ms']:.1f} ms")
    print(f"  P95:    {pt_results['p95_ms']:.1f} ms")
    print(f"  P99:    {pt_results['p99_ms']:.1f} ms")
    print(f"  Std:    {pt_results['std_ms']:.1f} ms")
    print(f"  Runs:   {pt_results['runs']}")
    
    # ── ONNX Latency (deployment target) ──
    if onnx_path is not None and os.path.exists(onnx_path):
        print()
        print("Inference Latency (ONNX Runtime, CPU):")
        print("-" * 40)
        # Tokenize benchmark texts
        encoded = allergen_tokenizer(benchmark_texts, padding=True, truncation=True, max_length=221, return_tensors="pt")
        ort_results = benchmark_onnx_latency(
            onnx_path=onnx_path,
            input_ids=encoded["input_ids"].numpy(),
            attention_mask=encoded["attention_mask"].numpy(),
            n_runs=100,
        )
        print(f"  Mean:   {ort_results['mean_ms']:.1f} ms")
        print(f"  Median: {ort_results['median_ms']:.1f} ms")
        print(f"  P95:    {ort_results['p95_ms']:.1f} ms")
        print(f"  P99:    {ort_results['p99_ms']:.1f} ms")
        print(f"  Std:    {ort_results['std_ms']:.1f} ms")
        print(f"  Runs:   {ort_results['runs']}")
        
        # Compare
        ratio = ort_results['mean_ms'] / pt_results['mean_ms']
        print(f"\n📊 ONNX / PyTorch speed ratio: {ratio:.2f}x")
    else:
        print("\n(Skipping ONNX benchmark — no ONNX model available.)")
else:
    print("Skipping benchmark — no model loaded.")

Inference Latency (PyTorch, CPU):
----------------------------------------
  Mean:   19.2 ms
  Median: 19.0 ms
  P95:    20.6 ms
  P99:    21.5 ms
  Std:    0.8 ms
  Runs:   100

Inference Latency (ONNX Runtime, CPU):
----------------------------------------
  Mean:   11.2 ms
  Median: 11.2 ms
  P95:    11.5 ms
  P99:    11.5 ms
  Std:    0.1 ms
  Runs:   100

📊 ONNX / PyTorch speed ratio: 0.59x


## 7. Deployment Report

Generate a summary report of the deployment readiness status. The report captures
ONNX export status, validation results, and latency benchmarks rather than TFLite
metrics (since the Android target uses ONNX Runtime Mobile).

In [8]:
# Build deployment report
onnx_exported = onnx_path is not None and os.path.exists(onnx_path)

# Track validation results explicitly instead of using fragile in-dir() checks
onnx_validated = False
if onnx_exported:
    try:
        onnx_validated = val.get("within_tolerance", False) if isinstance(val, dict) else False
    except NameError:
        onnx_validated = False

report = {
    "deployment_date": pd.Timestamp.now().isoformat(),
    "deployment_target": "ONNX Runtime Mobile (Android)",
    "allergen_model": {
        "source": allergen_model_path if os.path.exists(allergen_model_path) else None,
        "onnx_exported": onnx_exported,
        "onnx_validated": onnx_validated,
        "tflite_exported": False,  # not used — replaced by ONNX Runtime Mobile
    },
    "semantic_model": {
        "source": semantic_model_path if os.path.exists(semantic_model_path) else None,
        "onnx_exported": os.path.exists(os.path.join(export_dir, "allergen_model", "allergen_model.onnx")),
        "tflite_exported": False,
    },
    "dependencies": check_deployment_requirements(),
}

# Add latency results if available (use explicit tracking instead of in-dir())
latency_results = {}
if onnx_exported:
    try:
        latency_results["latency_pytorch_cpu"] = {k: pt_results[k] for k in ["mean_ms", "median_ms", "p95_ms", "std_ms"]}
    except NameError:
        latency_results["latency_pytorch_cpu"] = None
    try:
        latency_results["latency_onnx_cpu"] = {k: ort_results[k] for k in ["mean_ms", "median_ms", "p95_ms", "std_ms"]}
    except NameError:
        latency_results["latency_onnx_cpu"] = None
    report.update({k: v for k, v in latency_results.items() if v is not None})

report_path = os.path.join(export_dir, "deployment_report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"Deployment report saved to {report_path}")
print()
print("📋 Next step: Run 10_mobile_benchmark.ipynb for Android integration testing")
print("   (uses ONNX Runtime Mobile — not TFLite)")

Deployment report saved to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/exported/deployment_report.json

📋 Next step: Run 10_mobile_benchmark.ipynb for Android integration testing
   (uses ONNX Runtime Mobile — not TFLite)
